In [ ]:
import pandas as pd
import math
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

from ragas.metrics import numeric_metric
from ragas.metrics.result import MetricResult

c:\main\data_science\projects\dl_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = [
    {"expression": "(2 + 3) * (4 - 1)", "expected": 15},
    {"expression": "5 * (6 + 2)", "expected": 40},
    {"expression": "10 - (3 + 2)", "expected": 5},
]

df = pd.DataFrame(dataset)
df.to_csv("data/test_dataset.csv", index=False)

In [4]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    expression: str
    result: float
    log_file: str
    expected: float

# Простой инструмент для вычисления математических выражений
def safe_eval(expression: str) -> float:
    """Безопасное вычисление математического выражения через Python eval."""
    try:
        # Ограничиваем только математические операции
        allowed_names = {
            "abs": abs,
            "sum": sum,
        }
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return float(result)
    except Exception as e:
        return float("nan")

# Нода агента - решает математическую задачу
def math_agent_node(state: AgentState) -> AgentState:
    expression = state["expression"]
    
    # Вычисляем результат
    result = safe_eval(expression)
    
    log = f"Expression: {expression}\nComputed result: {result}"
    
    # Добавляем сообщение в историю
    message = AIMessage(
        content=f"Решил выражение '{expression}' = {result}\nЛог: {log}"
    )
    
    return {
        **state,
        "messages": [message],
        "result": result,
        "log_file": log,
    }

# Создаем граф
workflow = StateGraph(AgentState)

# Добавляем ноду
workflow.add_node("math_agent", math_agent_node)

# Устанавливаем входную точку и выход
workflow.set_entry_point("math_agent")
workflow.add_edge("math_agent", END)

# Компилируем агента
math_agent = workflow.compile()

In [5]:
@numeric_metric(name="correctness")
def correctness_metric(prediction: float, actual: float):
    """Calculate correctness of the prediction."""
    if isinstance(prediction, str) and "ERROR" in prediction:   
        return 0.0

    result = 1.0 if abs(prediction - actual) < 1e-5 else 0.0
    return MetricResult(value=result, reason=f"Prediction: {prediction}, Actual: {actual}")


In [6]:
from ragas import experiment

@experiment()
async def run_experiment(row):
    initial_state = {
        "messages": [],
        "expression": row["expression"],
        "expected": row["expected"],
        "result": 0.0,
        "log_file": ""
    }
    
    result = math_agent.invoke(initial_state)
    return {
        "result": result["result"],
        "log_file": result["log_file"]
    }

In [17]:
for row in dataset:
    expression = row["expression"]
    expected = row["expected"]

    prediction = await run_experiment(row)
    correctness = correctness_metric(prediction["result"], expected)

    print(f"\n📝 '{expression}'")
    print(f"✅ Ожидаемый: {expected}")
    print(f"🤖 Предсказание: {prediction['result']}")
    print(f"🎯 Корректность: {correctness}")
    print("-" * 50)


📝 '(2 + 3) * (4 - 1)'
✅ Ожидаемый: 15
🤖 Предсказание: 15.0
🎯 Корректность: MetricResult(value=1.0, reason='Prediction: 15.0, Actual: 15')
--------------------------------------------------

📝 '5 * (6 + 2)'
✅ Ожидаемый: 40
🤖 Предсказание: 40.0
🎯 Корректность: MetricResult(value=1.0, reason='Prediction: 40.0, Actual: 40')
--------------------------------------------------

📝 '10 - (3 + 2)'
✅ Ожидаемый: 5
🤖 Предсказание: 5.0
🎯 Корректность: MetricResult(value=1.0, reason='Prediction: 5.0, Actual: 5')
--------------------------------------------------
